# 02 — Distribuciones Conjuntas y Análisis Condicional

## 1. Marco Teórico: Distribución Normal Bivariada
La distribución normal bivariada describe el comportamiento conjunto de dos variables aleatorias continuas $X_1$ y $X_2$. Su función de densidad de probabilidad conjunta está dada por:
$$f(x_1, x_2) = \frac{1}{2\pi \sigma_1 \sigma_2 \sqrt{1-\rho^2}} \exp\left( -\frac{1}{2(1-\rho^2)} \left[ \frac{(x_1-\mu_1)^2}{\sigma_1^2} + \frac{(x_2-\mu_2)^2}{\sigma_2^2} - \frac{2\rho(x_1-\mu_1)(x_2-\mu_2)}{\sigma_1\sigma_2} \right] \right)$$
donde $\mu_1, \mu_2$ son las medias, $\sigma_1^2, \sigma_2^2$ son las varianzas y $\rho$ es el coeficiente de correlación de Pearson.

En forma matricial, si definimos el vector de variables $\mathbf{X} = [X_1, X_2]^T$, la distribución se denota como $\mathbf{X} \sim N_2(\boldsymbol{\mu}, \boldsymbol{\Sigma})$, con densidad:
$$f(\mathbf{x}) = \frac{1}{2\pi |\boldsymbol{\Sigma}|^{1/2}} \exp\left( -\frac{1}{2} (\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu}) \right)$$
donde $\boldsymbol{\mu}$ es el vector de medias y $\boldsymbol{\Sigma}$ es la matriz de covarianza. Las curvas de nivel donde la densidad es constante corresponden a elipses centradas en $\boldsymbol{\mu}$. La ecuación de estas elipses de contorno está definida por la distancia de Mahalanobis al cuadrado:
$$d^2(\mathbf{x}) = (\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu}) = c^2$$
Bajo el supuesto de normalidad bivariada, la distancia de Mahalanobis al cuadrado sigue una distribución Chi-cuadrado con 2 grados de libertad: $d^2(\mathbf{x}) \sim \chi^2_2$. Esto permite definir elipses de densidad para un nivel de probabilidad $\alpha$ (por ejemplo, 50%, 80% o 95%) utilizando el valor crítico $c^2 = \chi^2_2(1-\alpha)$.

## 2. Aplicación Práctica: Temperatura vs Humedad
Analizaremos conjuntamente la Temperatura y la Humedad Relativa estandarizadas ($Z$-scores). Desde una perspectiva física, estas variables presentan una relación termodinámica inversa estrecha (colinealidad física). Estandarizarlas nos permite modelar su interacción conjunta eliminando las diferencias de escala.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
os.chdir(ROOT)

import pandas as pd
import IPython.display as display
import importlib.util
spec = importlib.util.spec_from_file_location('joint', ROOT / 'Scripts/08_joint_distributions.py')
joint = importlib.util.module_from_spec(spec)
spec.loader.exec_module(joint)

df, pollution_q1, pollution_q3 = joint.prepare_data()
print(f"Dimensiones de los datos preparados: {df.shape[0]} filas")
print(f"Par bivariado bajo análisis: {joint.BIVAR_Z}")
print(f"Variables que componen el índice de contaminación ponderado Z: {joint.POLLUTION_Z}")

## 3. Estimación de Densidad por Kernel (KDE 2D) y Elipses Empíricas
La Estimación de Densidad por Kernel de dos dimensiones (KDE 2D) nos permite ver de manera no paramétrica cómo se distribuyen conjuntamente las observaciones de Temperatura y Humedad.

Luego, construimos las elipses empíricas a niveles de probabilidad del 50%, 80% y 95% calculadas a partir de la media y matriz de covarianza muestral. Estas elipses muestran dónde se concentra la mayor proporción de la masa de datos.

In [ ]:
bivar, conditional = joint.save_summaries(df, pollution_q1, pollution_q3)
print("=== ESTADÍSTICAS BIVARIADAS DE TEMPERATURA Y HUMEDAD Z ===")
display.display(bivar.round(3))

# Graficar KDE 2D y elipses empíricas
joint.plot_kde_2d(df)
joint.plot_density_ellipses(df)

from IPython.display import Image
display.display(Image(filename='Figures/Joint_Distributions/01_kde2d_temperatura_humedad.png'))
display.display(Image(filename='Figures/Joint_Distributions/02_elipses_densidad_temperatura_humedad.png'))

## 4. Comparación: KDE Observada vs. Distribución Normal Bivariada Teórica
Contrastamos visualmente las curvas de nivel (contornos) de la densidad KDE real observada en los datos (curvas rojas) frente a los contornos de una distribución normal bivariada teórica ajustada con los parámetros muestrales (curvas azules).

Si los datos fueran perfectamente normales bivariados, las curvas rojas y azules coincidirían perfectamente en orientación, centro y espaciamiento. Las diferencias revelan desviaciones de la normalidad conjunta (asimetrías, colas pesadas o curvaturas no lineales en los extremos).

In [ ]:
# Ajustar normal teórica vs observada
joint.plot_bivariate_normal(df)
Image(filename='Figures/Joint_Distributions/03_normal_bivariada_temperatura_humedad.png')

## 5. Análisis Condicional de la Biodiversidad por Niveles de Contaminación
Definimos un índice de contaminación agregando los contaminantes $PM_{2.5}$, $PM_{10}$, $NO_2$, $CO$, y $O_3$ estandarizados. A partir de sus cuartiles extremos, segregamos los datos en dos escenarios de estrés ambiental:
- **Baja Contaminación ($Q1$):** Observaciones con niveles inferiores al percentil 25.
- **Alta Contaminación ($Q3$):** Observaciones con niveles superiores al percentil 75.

Evaluamos la densidad condicional de nuestro índice ecológico (Shannon_Index) dada esta partición ambiental. Esto nos permite comprobar visual y estadísticamente si existe un desplazamiento de la distribución de biodiversidad ante la presencia de estrés por contaminación.

In [ ]:
print(f"=== UMBRALES DE CONTAMINACIÓN Z ===")
print(f"Baja Contaminación (<= Q1): {pollution_q1:.3f}")
print(f"Alta Contaminación (>= Q3): {pollution_q3:.3f}")
print("\n=== RESUMEN CONDICIONAL DEL ÍNDICE DE SHANNON ===")
display.display(conditional.round(3))

# Graficar distribución condicional
joint.plot_conditional_shannon(df)
Image(filename='Figures/Joint_Distributions/04_shannon_condicional_contaminacion.png')

## Conclusiones e Interpretación Ecológica
1. **Colinealidad Física Clima:** La Temperatura y la Humedad Relativa muestran una elipse con pendiente negativa muy marcada, indicando una correlación negativa intensa. Físicamente, el aire bogotano a mayor temperatura se vuelve más seco, y el enfriamiento nocturno o por lluvias eleva la humedad.
2. **Ajuste Bivariado:** La forma elíptica de la densidad observada sigue razonablemente la normal teórica en la región central, pero presenta colas más extendidas (asimetría conjunta y curtosis) en los extremos, lo que afectará los supuestos de las pruebas paramétricas multivariadas clásicas.
3. **Efecto de la Contaminación:** Se aprecia visualmente que en condiciones de Alta Contaminación (Q3, curva roja), la distribución del índice de Shannon se desplaza hacia valores menores de diversidad de aves en comparación con el escenario de Baja Contaminación (Q1, curva verde). Además, la dispersión del índice cambia, sugiriendo heterocedasticidad condicional.